# SoilGrids clay % (0–30 cm)

**Preferred batch path:**

```bash
python transformation/soilgrids/extract_clay.py --site plymouth
```

OpenLandMap clay weight fraction (0–30 cm mean) → `layers.clay_pct` under `landslide_hazard/sites/<city>/data/input/`.


OpenLandMap clay weight fraction averaged over 0–30 cm for `LANDSLIDES_SITE`. Used as the cohesion / pore-pressure amplifier in landslide hazard.


In [ ]:
# Site configuration — transformation/landslide_hazard city configs + model defaults
import os
import sys
from pathlib import Path

_HERE = Path.cwd().resolve()
_LANDSLIDE_HAZARD = None
for _candidate in [_HERE, *_HERE.parents]:
    _probe = _candidate / "landslide_hazard" if _candidate.name != "landslide_hazard" else _candidate
    if (_probe / "site_config.py").is_file() and (_probe / "config" / "sites").is_dir():
        _LANDSLIDE_HAZARD = _probe
        break
if _LANDSLIDE_HAZARD is None:
    raise FileNotFoundError("Could not locate transformation/landslide_hazard from notebook cwd")

sys.path.insert(0, str(_LANDSLIDE_HAZARD))
from site_config import load_site_config

LANDSLIDE_HAZARD_ROOT = _LANDSLIDE_HAZARD

# Set the city here (edit this line). That value wins for interactive runs.
# Use None to fall back to env LANDSLIDES_SITE (default porto_alegre).
SITE_SLUG = "plymouth"  # or: "porto_alegre" | "edina" | "richfield" | "rochester" | "apple_valley" | None
if SITE_SLUG is None:
    SITE_SLUG = os.environ.get("LANDSLIDES_SITE", "porto_alegre")
SITE_CONFIG = load_site_config(SITE_SLUG, LANDSLIDE_HAZARD_ROOT)
SITE_ROOT = SITE_CONFIG["paths_abs"]["site_root"]
INPUT_DIR = SITE_CONFIG["paths_abs"]["data_input"]
INTERMEDIATE_DIR = SITE_CONFIG["paths_abs"]["data_intermediate"]
OUTPUT_DIR = SITE_CONFIG["paths_abs"]["data_output"]
OUT_ROOT = SITE_CONFIG["paths_abs"]["out"]
CACHE_DIR = SITE_CONFIG["paths_abs"]["cache"]
STYLES_DIR = SITE_CONFIG["paths_abs"]["styles"]
OUTPUT_PREFIX = SITE_CONFIG["output_prefix"]
SEASON = SITE_CONFIG["season"]
SEASON_LABEL = SITE_CONFIG["season_label"]
START_YEAR = int(SITE_CONFIG["start_year"])
END_YEAR = int(SITE_CONFIG["end_year"])
DW_YEAR = int(SITE_CONFIG.get("dw_year", 2023))
HAZARD_CFG = SITE_CONFIG["hazard"]
PUBLISH_CFG = SITE_CONFIG.get("publish", {})
BAIRRO_CFG = SITE_CONFIG.get("bairro", {})
MODEL_CONFIG_PATH = SITE_CONFIG["model_config_path"]
LAYER_FILES = SITE_CONFIG["layers"]
OUTPUT_FILES = SITE_CONFIG["outputs"]

for _p in (INPUT_DIR, INTERMEDIATE_DIR, OUTPUT_DIR, OUT_ROOT, CACHE_DIR, STYLES_DIR):
    Path(_p).mkdir(parents=True, exist_ok=True)

print(f"Landslide hazard site: {SITE_CONFIG['display_name']} ({SITE_SLUG})")
print(f"Config: {SITE_CONFIG['config_path']}")
print(f"Model defaults: {MODEL_CONFIG_PATH}")
print(f"Season: {SEASON_LABEL} {START_YEAR}-{END_YEAR}")
print(f"Inputs -> {INPUT_DIR}")


## 0. Earth Engine + ROI


In [ ]:
import ee
ee.Initialize(project='eecc-maureen')

# Site ROI: use the site polygon when available; fall back to bbox.
import json
import ee


def load_site_roi() -> ee.Geometry:
    boundary_path = SITE_CONFIG["boundary_path_abs"]
    if boundary_path.exists():
        data = json.loads(boundary_path.read_text())
        if data.get("type") == "FeatureCollection":
            features = [
                ee.Feature(ee.Geometry(feature["geometry"]), feature.get("properties", {}))
                for feature in data.get("features", [])
                if feature.get("geometry")
            ]
            if features:
                return ee.FeatureCollection(features).geometry()
        if data.get("type") == "Feature":
            return ee.Geometry(data["geometry"])
        if data.get("type") in {"Polygon", "MultiPolygon", "GeometryCollection"}:
            return ee.Geometry(data)
    return ee.Geometry.Rectangle(SITE_CONFIG["bbox"])


roi = load_site_roi()
print(f"ROI loaded for {SITE_CONFIG['display_name']} from {SITE_CONFIG['boundary_path_abs']}")


## 1. Export clay %


In [ ]:
# OpenLandMap SoilGrids clay weight fraction, 0–30 cm mean → landslide site input/
from gee_local_export import export_image_to_input

clay_raw = (
    ee.Image("OpenLandMap/SOL/SOL_CLAY-WFRACTION_USDA-3A1A1A_M/v02")
    .select(["b0", "b10", "b30"])
    .reduce(ee.Reducer.mean())
    .divide(10)
    .rename("clay_pct")
    .clip(roi)
    .reproject(crs="EPSG:4326", scale=250)
    .toFloat()
)

stats = clay_raw.reduceRegion(
    reducer=ee.Reducer.percentile([0, 50, 90, 100]),
    geometry=roi,
    scale=250,
    maxPixels=1e9,
).getInfo()
print("Clay % stats (0–30 cm):", stats)

export_image_to_input(
    clay_raw,
    filename=LAYER_FILES["clay_pct"],
    region=roi,
    scale=250,
    input_dir=INPUT_DIR,
    crs="EPSG:4326",
    description=Path(LAYER_FILES["clay_pct"]).stem,
    drive_folder="gee_exports",
)
print("Exports complete. Files land under:", INPUT_DIR)


## 2. Inspect


In [ ]:
import numpy as np
import rasterio
import matplotlib.pyplot as plt

clay_tif = INPUT_DIR / LAYER_FILES['clay_pct']
assert clay_tif.exists(), f'File not found: {clay_tif}'

with rasterio.open(clay_tif) as src:
    arr = src.read(1)
    nd = src.nodata
    print(f'Shape: {src.height}×{src.width}  CRS={src.crs}  res={src.res}')

valid = arr[np.isfinite(arr)]
if nd is not None:
    valid = valid[valid != nd]
print(f'Clay %: min={valid.min():.1f} mean={valid.mean():.1f} max={valid.max():.1f}')
print(f'% pixels > 40% clay (formula saturation): {(valid > 40).mean()*100:.1f}%')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(valid, bins=40, color='saddlebrown', edgecolor='none')
axes[0].axvline(40, color='red', linestyle='--', label='40% sat')
axes[0].legend(); axes[0].set_title(f"{SITE_CONFIG['display_name']} — clay %")
im = axes[1].imshow(arr, cmap='YlOrBr'); fig.colorbar(im, ax=axes[1], fraction=0.046)
plt.tight_layout(); plt.show()


## 3. Publish


In [ ]:
# Publish local input GeoTIFF → COG + colorized XYZ tiles + value-encoded XYZ tiles.
from pathlib import Path
import shutil
import subprocess
import numpy as np
import rasterio

in_tif = INPUT_DIR / LAYER_FILES["clay_pct"]
slug = Path(LAYER_FILES["clay_pct"]).stem
out_dir = OUT_ROOT / slug
colors_txt = STYLES_DIR / "clay_pct_250m_colors.txt"
out_dir.mkdir(parents=True, exist_ok=True)

cog_tif = out_dir / f"{slug}_cog.tif"
colorized_tif = out_dir / f"{slug}_colorized.tif"
value_encoded_tif = out_dir / f"{slug}_value_encoded_rgb.tif"
tiles_dir = out_dir / "tiles_visual"
value_tiles_dir = out_dir / "tiles_values"
decode_txt = out_dir / f"{slug}_value_tiles_decode.txt"

print(f"Input: {in_tif}")
print(f"Output dir: {out_dir}")
assert in_tif.exists(), f"Missing input raster: {in_tif}"
assert colors_txt.exists(), f"Missing color table: {colors_txt}"

subprocess.run([
    "gdal_translate", str(in_tif), str(cog_tif),
    "-of", "COG", "-co", "COMPRESS=DEFLATE", "-co", "PREDICTOR=2",
], check=True)
print(f"Created COG: {cog_tif}")

subprocess.run([
    "gdaldem", "color-relief", str(in_tif), str(colors_txt), str(colorized_tif),
    "-alpha", "-co", "COMPRESS=LZW",
], check=True)
print(f"Created colorized raster: {colorized_tif}")

if tiles_dir.exists():
    shutil.rmtree(tiles_dir)
subprocess.run([
    "gdal2tiles.py", "-z", str(PUBLISH_CFG.get("tile_zoom", "8-15")),
    "--processes=4", str(colorized_tif), str(tiles_dir),
], check=True)
print(f"Visual tiles: {tiles_dir}")

# Value encoding: float×10000
with rasterio.open(in_tif) as src:
    arr = src.read(1).astype(np.float32)
    profile = src.profile.copy()
    nodata = src.nodata

if np.issubdtype(arr.dtype, np.floating):
    encoded = np.clip(np.rint(np.nan_to_num(arr, nan=0.0) * 10000), 0, 65535).astype(np.uint16)
    decode = "encoded_int = R + 256*G\nvalue = encoded_int / 10000.0\n"
else:
    encoded = np.clip(arr, 0, 65535).astype(np.uint16)
    decode = "encoded_int = R + 256*G\nvalue = encoded_int\n"

r = (encoded & 0xFF).astype(np.uint8)
g = ((encoded >> 8) & 0xFF).astype(np.uint8)
b = np.zeros_like(r)
profile.update(count=3, dtype="uint8", nodata=None, compress="lzw")
with rasterio.open(value_encoded_tif, "w", **profile) as dst:
    dst.write(r, 1)
    dst.write(g, 2)
    dst.write(b, 3)
print(f"Value-encoded RGB: {value_encoded_tif}")

if value_tiles_dir.exists():
    shutil.rmtree(value_tiles_dir)
subprocess.run([
    "gdal2tiles.py", "-z", str(PUBLISH_CFG.get("tile_zoom", "8-15")),
    "--processes=4", str(value_encoded_tif), str(value_tiles_dir),
], check=True)
decode_txt.write_text(decode)
print(f"Value tiles: {value_tiles_dir}")
print(f"Decode metadata: {decode_txt}")
